# Crop-yield paper: data extension and field-image check

Run the cells top to bottom in Google Colab. **Google Drive is used for everything that should survive a restart**: the FAOSTAT downloads, the PlantVillage hash cache, the downloaded field datasets and all results. After the first run, a restart costs seconds, not minutes.

1. **Download FAOSTAT directly** (Pakistan wheat and maize yield, national pesticide use, temperature change) and build a longer crop-year table.
2. **Re-run the honest evaluation** (crop mean, linear trend, learned models, trend + residual model) and check the pesticide series and its FAO flags.
3. **Test the leaf classifier on images it has not seen**, after removing every image that is a near-duplicate of a PlantVillage training image.
4. **Measure false alarms on healthy field leaves** from other public datasets (no photos needed from you).

**Set-up (once):** make a folder `crop_yield_project` in your Google Drive and put these files in it: `leaf_classifier.h5` (required), `kaggle.json` (needed for the PlantVillage download), and optionally `crop_year_dataset.csv` (from the paper's `code/` folder).

**Why was PlantVillage slow before?** The Kaggle zip holds three copies of every image (colour, grayscale, segmented), about 160,000 files, and the old cell unpacked and hashed all of them. The new cell extracts only the 3,852 colour maize images, hashes them in parallel, and saves the hashes to Drive, so the download happens at most once.

In [ ]:
# ---- 0. Setup: Google Drive, folders ------------------------------------------------
!pip -q install xgboost imagehash kaggle

import os, io, zipfile, warnings, json, re, glob, pickle
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import requests
warnings.filterwarnings("ignore")

USE_DRIVE = True
try:
    if not USE_DRIVE: raise ImportError("Drive disabled")
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/crop_yield_project"
except Exception as e:
    print("Drive not mounted (", e, ") -> using local /content/crop_yield_project (nothing will survive a restart)")
    ROOT = "/content/crop_yield_project"

DATA_DIR  = f"{ROOT}/data";               # FAOSTAT zips (cached)
OUT_DIR   = f"{ROOT}/results";            # everything the paper needs
CACHE_DIR = f"{ROOT}/cache";              # PlantVillage hashes etc.
ZIP_DIR   = f"{ROOT}/field_extra_zips";   # Mendeley / Kaggle downloads (zips)
LOCAL     = "/content/work"               # fast local disk for unpacked images
for d in [ROOT, DATA_DIR, OUT_DIR, CACHE_DIR, ZIP_DIR, LOCAL]: os.makedirs(d, exist_ok=True)

def find_file(name):
    for d in [ROOT, "/content", "/content/drive/MyDrive"]:
        p = os.path.join(d, name)
        if os.path.exists(p): return p
    return None

kj = find_file("kaggle.json")
if kj:
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    !cp "$kj" ~/.kaggle/kaggle.json && chmod 600 ~/.kaggle/kaggle.json
    print("kaggle.json found:", kj)
else:
    print("kaggle.json not found (put it in", ROOT, "); needed only for the PlantVillage download")

AREA = "Pakistan"; FIRST_YEAR = 1990; SEED = 42
print("ROOT =", ROOT)

## Part 1. FAOSTAT download

FAOSTAT publishes every dataset as a bulk zip. The cell below tries two known URL patterns. If both fail (FAO sometimes renames files), open <https://www.fao.org/faostat/en/#data>, download the three normalized CSV zips by hand, put them in `/content/data`, and re-run the cell.

In [ ]:
import requests

FILES = {
    "crops": "Production_Crops_Livestock_E_All_Data_(Normalized).zip",
    "pest":  "Inputs_Pesticides_Use_E_All_Data_(Normalized).zip",
    "temp":  "Environment_Temperature_change_E_All_Data_(Normalized).zip",
}
BASES = ["https://bulks-faostat.fao.org/production/",
         "https://fenixservices.fao.org/faostat/static/bulkdownloads/"]

def fetch(name):
    path = os.path.join(DATA_DIR, name)
    if os.path.exists(path):
        print("found", name); return path
    for base in BASES:
        try:
            print("downloading", base + name)
            r = requests.get(base + name, timeout=300); r.raise_for_status()
            open(path, "wb").write(r.content); return path
        except Exception as e:
            print("   failed:", e)
    raise RuntimeError(f"Could not download {name}. Download it manually from FAOSTAT into {DATA_DIR}.")

def load_area(path, area=AREA):
    z = zipfile.ZipFile(path)
    name = [n for n in z.namelist() if "All_Data" in n and n.endswith(".csv") and "Flags" not in n and "Symbols" not in n][0]
    parts = []
    for ch in pd.read_csv(z.open(name), encoding="latin-1", chunksize=200_000, low_memory=False):
        parts.append(ch[ch["Area"] == area])
    return pd.concat(parts, ignore_index=True)

raw = {k: load_area(fetch(v)) for k, v in FILES.items()}
for k, v in raw.items():
    print(k, v.shape, "| columns:", list(v.columns)[:14])

In [ ]:
# ---- Build the crop-year table ----------------------------------------------
def to_ton_per_ha(value, unit):
    u = str(unit).lower().replace(" ", "")
    if u in ("hg/ha", "100g/ha"): return value / 10_000
    if u == "kg/ha":              return value / 1_000
    if u in ("t/ha", "tonnes/ha"): return value
    raise ValueError(f"unexpected yield unit {unit!r}")

c = raw["crops"]
c = c[(c["Element"].str.lower() == "yield") & (c["Item"].str.lower().isin(["wheat", "maize (corn)", "maize"]))].copy()
c["crop"] = np.where(c["Item"].str.lower().str.startswith("wheat"), "wheat", "maize")
c["yield_ton_ha"] = [to_ton_per_ha(v, u) for v, u in zip(c["Value"], c["Unit"])]
yields = c[["crop", "Year", "yield_ton_ha", "Flag"]].rename(columns={"Year": "year", "Flag": "yield_flag"})

p = raw["pest"]
p = p[(p["Item"].str.lower().str.contains("pesticides (total)", regex=False)) & (p["Element"].str.lower().str.contains("use"))].copy()
if p.empty:   # fall back to any "total" pesticide item
    p = raw["pest"][raw["pest"]["Item"].str.lower().str.contains("total")].copy()
pest = p.groupby("Year", as_index=False).agg(pesticides_tonnes=("Value", "first"), pest_flag=("Flag", "first")).rename(columns={"Year": "year"})

t = raw["temp"]
t = t[(t["Element"].str.lower() == "temperature change")]
if "Months" in t.columns:
    t = t[t["Months"].str.lower().str.contains("meteorological")]
temp = t.groupby("Year", as_index=False).agg(temp_change=("Value", "first")).rename(columns={"Year": "year"})

T = yields.merge(pest, on="year", how="left").merge(temp, on="year", how="left")
T = T[T.year >= FIRST_YEAR].sort_values(["crop", "year"]).reset_index(drop=True)
T["crop_code"] = (T.crop == "wheat").astype(int)
print("rows:", len(T), "| years:", T.year.min(), "-", T.year.max(), "| missing pesticides:", T.pesticides_tonnes.isna().sum(), "| missing temp:", T.temp_change.isna().sum())
T.to_csv(f"{OUT_DIR}/extended_crop_year_table.csv", index=False)
display(T.groupby("crop").agg(years=("year", "nunique"), first=("year", "min"), last=("year", "max"), mean_yield=("yield_ton_ha", "mean")))
display(T.head(8))

### Checks on the new table
Two checks the paper needs: (a) do FAO's yields agree with the Kaggle table we used before, and (b) is the pesticide series still a straight line, and what do FAO's own flags say about it? In FAOSTAT, flag `I` means *imputed* and `E` means *estimated*, so this settles the interpolation question directly.

In [ ]:
# (a) agreement with the earlier Kaggle-derived table (optional file)
CY = find_file("crop_year_dataset.csv")
if CY:
    old = pd.read_csv(CY)[["crop", "year", "yield_ton_ha"]].rename(columns={"yield_ton_ha": "old"})
    m = old.merge(T[["crop", "year", "yield_ton_ha"]], on=["crop", "year"])
    print("overlapping crop-years:", len(m), "| max abs difference (Ton/Ha):", float((m.old - m.yield_ton_ha).abs().max()))
else:
    print("(skipped: put crop_year_dataset.csv in your crop_yield_project Drive folder to run this check)")

# (b) pesticide series: longest exact straight-line run, and FAO flags
ps = pest[pest.year >= FIRST_YEAR].sort_values("year").reset_index(drop=True)
d1 = np.diff(ps.pesticides_tonnes.values)
best = (0, 0, 0); start = 0
for i in range(1, len(d1) + 1):
    if i == len(d1) or abs(d1[i] - d1[i - 1]) > 1.0:   # slope changes by more than 1 tonne/yr
        if i - start > best[0]: best = (i - start, start, i)
        start = i
n, a, b = best
print(f"longest straight-line run: {n+1} years, {int(ps.year[a])} to {int(ps.year[min(b, len(ps)-1)])}, slope {d1[a]:.1f} t/yr")
print("flags in series:", ps.pest_flag.fillna("(none)").value_counts().to_dict())
print(ps.tail(12).to_string(index=False))

fig, ax = plt.subplots(1, 3, figsize=(11, 3))
for cr, g in T.groupby("crop"): ax[0].plot(g.year, g.yield_ton_ha, "o-", ms=3, label=cr)
ax[0].set_title("Yield (Ton/Ha)"); ax[0].legend()
ax[1].plot(ps.year, ps.pesticides_tonnes, "o-", ms=3); ax[1].set_title("Pesticide use (t)")
for y, f in zip(ps.year, ps.pest_flag):
    if isinstance(f, str): ax[1].annotate(f, (y, ps.set_index("year").pesticides_tonnes[y]), fontsize=7)
ax[2].plot(temp.year, temp.temp_change, "o-", ms=3); ax[2].set_title("Temperature change (°C)")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/ext_series.png", dpi=150); plt.show()

## Part 2. Honest evaluation on the longer table

Same protocols and settings as the paper: leave-one-year-out (LOYO) and forward chaining (test years from `FIRST_YEAR + 8`), RMSE with a bootstrap interval over test years. Feature set A' = pesticide use, temperature change, crop; C' adds the year. The residual model fits the per-crop trend on the training years and learns what the trend misses.

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

D = T.dropna(subset=["yield_ton_ha", "pesticides_tonnes", "temp_change"]).reset_index(drop=True)
print("modelling table:", D.shape, "years", D.year.min(), "-", D.year.max())

def make(name):
    if name == "Ridge": return make_pipeline(StandardScaler(), Ridge(alpha=1.0))
    if name == "RF":    return RandomForestRegressor(n_estimators=200, max_depth=8, random_state=SEED, n_jobs=-1)
    if name == "XGB":   return XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=SEED, verbosity=0)
    if name == "SVR":   return make_pipeline(StandardScaler(), SVR(kernel="rbf", C=100, gamma=0.1, epsilon=0.1))
def make_small(name):   # for residual models: deliberately small, chosen once, not tuned
    if name == "Ridge": return make_pipeline(StandardScaler(), Ridge(alpha=1.0))
    if name == "RF":    return RandomForestRegressor(n_estimators=200, max_depth=3, min_samples_leaf=3, random_state=SEED, n_jobs=-1)
    if name == "XGB":   return XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=2, random_state=SEED, verbosity=0)
    if name == "SVR":   return make_pipeline(StandardScaler(), SVR(C=1, gamma="scale", epsilon=0.05))

def trend_fit_predict(tr, te):
    out = np.zeros(len(te)); fit = np.zeros(len(tr))
    for cr in D.crop.unique():
        a = tr[tr.crop == cr]; lr = LinearRegression().fit(a[["year"]], a.yield_ton_ha)
        out[(te.crop == cr).values] = lr.predict(te.loc[te.crop == cr, ["year"]])
        fit[(tr.crop == cr).values] = lr.predict(a[["year"]])
    return out, fit

def folds(protocol):
    yrs = np.sort(D.year.unique())
    tests = yrs if protocol == "LOYO" else yrs[yrs >= FIRST_YEAR + 8]
    for y in tests:
        tr = D[D.year != y] if protocol == "LOYO" else D[D.year < y]
        yield tr, D[D.year == y]

def run(predict_fn, protocol):
    yy, pp, yr = [], [], []
    for tr, te in folds(protocol):
        p = predict_fn(tr, te); yy += list(te.yield_ton_ha); pp += list(p); yr += list(te.year)
    yy, pp, yr = map(np.array, (yy, pp, yr))
    rng = np.random.default_rng(0); ys = np.unique(yr); bs = []
    for _ in range(2000):
        pick = rng.choice(ys, len(ys)); idx = np.concatenate([np.where(yr == a)[0] for a in pick])
        bs.append(np.sqrt(np.mean((yy[idx] - pp[idx]) ** 2)))
    rmse = np.sqrt(np.mean((yy - pp) ** 2)); r2 = 1 - np.sum((yy - pp) ** 2) / np.sum((yy - yy.mean()) ** 2)
    return dict(rmse=rmse, ci_lo=np.percentile(bs, 2.5), ci_hi=np.percentile(bs, 97.5), r2=r2), (yy, pp, yr)

A = ["pesticides_tonnes", "temp_change", "crop_code"]; C = A + ["year"]
def direct(name, feats):  return lambda tr, te: make(name).fit(tr[feats], tr.yield_ton_ha).predict(te[feats])
def crop_mean(tr, te):    return te.crop.map(tr.groupby("crop").yield_ton_ha.mean()).values
def trend_only(tr, te):   return trend_fit_predict(tr, te)[0]
def resid(name, feats):
    def f(tr, te):
        tp, fit = trend_fit_predict(tr, te)
        return tp + make_small(name).fit(tr[feats], tr.yield_ton_ha.values - fit).predict(te[feats])
    return f

rows = []
specs = [("Crop mean", crop_mean), ("Crop linear trend", trend_only)]
specs += [(f"{m} (A')", direct(m, A)) for m in ["Ridge", "RF", "XGB", "SVR"]]
specs += [(f"{m} + year (C')", direct(m, C)) for m in ["Ridge", "RF", "XGB", "SVR"]]
specs += [(f"Trend + {m} residual", resid(m, A)) for m in ["Ridge", "RF", "XGB", "SVR"]]
for name, fn in specs:
    for prot in ["LOYO", "Forward"]:
        r, _ = run(fn, prot); rows.append(dict(model=name, protocol=prot, **r))
res = pd.DataFrame(rows); res.to_csv(f"{OUT_DIR}/ext_protocol_results.csv", index=False)
tab = res.pivot(index="model", columns="protocol", values=["rmse", "ci_lo", "ci_hi", "r2"]).round(3)
display(tab.loc[[s[0] for s in specs]])

In [ ]:
# ---- Which residual features matter? (paired bootstrap against the trend) -----
def paired_gain(feats, protocol, name="RF"):
    _, (yy, pt, yr) = run(trend_only, protocol); _, (_, pm, _) = run(resid(name, feats), protocol)
    rng = np.random.default_rng(1); ys = np.unique(yr); g = []
    for _ in range(2000):
        pick = rng.choice(ys, len(ys)); idx = np.concatenate([np.where(yr == a)[0] for a in pick])
        g.append(np.sqrt(np.mean((yy[idx] - pt[idx]) ** 2)) - np.sqrt(np.mean((yy[idx] - pm[idx]) ** 2)))
    return np.mean(g), np.percentile(g, 2.5), np.percentile(g, 97.5)

subsets = {"crop only": ["crop_code"], "+ year": ["crop_code", "year"], "+ temperature change": ["crop_code", "temp_change"],
           "+ pesticide use": ["crop_code", "pesticides_tonnes"], "+ both": A}
ab = []
for lab, f in subsets.items():
    for prot in ["LOYO", "Forward"]:
        g, lo, hi = paired_gain(f, prot); ab.append(dict(features=lab, protocol=prot, gain=g, lo=lo, hi=hi))
ab = pd.DataFrame(ab); ab.to_csv(f"{OUT_DIR}/ext_residual_ablation.csv", index=False)
display(ab.round(3))

## Helpers for the image tests (run this cell every session; it is quick)

In [ ]:
import imagehash, tensorflow as tf
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

THRESH = 6      # max differing bits (of 64) for two images to count as the same picture

def list_images(root, must_contain=None):
    out = []
    for dp, _, fs in os.walk(root):
        for f in fs:
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                full = os.path.join(dp, f)
                if must_contain is None or any(k in full.lower() for k in must_contain): out.append(full)
    return out

def _open_small(path, size):
    im = Image.open(path)
    if im.format == "JPEG": im.draft("RGB", (size, size))     # fast decode of big photos
    return im.convert("RGB")

def phash_int(path):
    h = imagehash.phash(_open_small(path, 256).resize((128, 128)), hash_size=8)
    return int("".join("1" if b else "0" for b in h.hash.flatten()), 2)

_HC_PATH = f"{CACHE_DIR}/hash_cache.pkl"
_HC = pickle.load(open(_HC_PATH, "rb")) if os.path.exists(_HC_PATH) else {}
def hash_many(paths, workers=4):
    todo = [p for p in paths if p not in _HC]
    if todo:
        with ThreadPoolExecutor(workers) as ex:
            for p, h in zip(todo, ex.map(phash_int, todo)): _HC[p] = h
        pickle.dump(_HC, open(_HC_PATH, "wb"))
    return np.array([_HC[p] for p in paths], dtype=np.uint64)

def popcount64(x):
    return np.unpackbits(np.ascontiguousarray(x).view(np.uint8).reshape(-1, 8), axis=1).sum(axis=1)

def min_dist_to(ref, hs):
    return np.array([popcount64(np.bitwise_xor(ref, h)).min() for h in hs])

CLASSIFIER = find_file("leaf_classifier.h5")
assert CLASSIFIER, f"Put leaf_classifier.h5 in {ROOT}"
model = tf.keras.models.load_model(CLASSIFIER)

def _load(p): return np.asarray(_open_small(p, 448).resize((224, 224)), dtype="float32") / 255.0
def predict_paths(paths, bs=64):
    out = []
    with ThreadPoolExecutor(4) as ex:
        for i in range(0, len(paths), bs):
            out += list(model.predict(np.stack(list(ex.map(_load, paths[i:i + bs]))), verbose=0).ravel())
    return np.array(out)

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    ph = k / n; c = (ph + z*z/(2*n)) / (1 + z*z/n); h = z * np.sqrt(ph*(1-ph)/n + z*z/(4*n*n)) / (1 + z*z/n)
    return c - h, c + h
print("helpers ready; classifier:", CLASSIFIER)

## PlantVillage maize hashes (cached on Drive)
First run: downloads the PlantVillage zip (about 1 to 2 GB, one to a few minutes), extracts **only the colour maize images**, hashes them and saves `cache/pv_maize_hashes.npy`. Every later run loads the cache in a second.

If you already have the PlantVillage maize images somewhere (for example in Drive from the original project), set `PV_LOCAL_DIR` to that folder and nothing is downloaded.

In [ ]:
PV_SLUG = "abdallahalidev/plantvillage-dataset"
PV_LOCAL_DIR = None        # e.g. "/content/drive/MyDrive/CropYieldProject/plantvillage"  (optional)
PV_CACHE = f"{CACHE_DIR}/pv_maize_hashes.npy"

if os.path.exists(PV_CACHE):
    pv_h = np.load(PV_CACHE); print("loaded cached PlantVillage hashes:", len(pv_h))
else:
    if PV_LOCAL_DIR and os.path.isdir(PV_LOCAL_DIR):
        pv_paths = list_images(PV_LOCAL_DIR, must_contain=["corn", "maize"])
    else:
        ZIPD = "/content/pv_zip"; PVD = f"{LOCAL}/plantvillage_maize"
        if not glob.glob(f"{ZIPD}/*.zip"):
            !kaggle datasets download -d $PV_SLUG -p $ZIPD
        zp = glob.glob(f"{ZIPD}/*.zip")[0]
        with zipfile.ZipFile(zp) as z:
            names = z.namelist()
            sel = [n for n in names if n.lower().endswith((".jpg", ".jpeg", ".png")) and re.search(r"corn|maize", n.lower()) and "/color/" in n.lower()]
            if not sel:   # folder layout differs: take every maize image and warn
                sel = [n for n in names if n.lower().endswith((".jpg", ".jpeg", ".png")) and re.search(r"corn|maize", n.lower())]
                print("WARNING: no '/color/' folder found; using all maize images:", len(sel))
            z.extractall(PVD, members=sel)
        pv_paths = list_images(PVD)
    print("PlantVillage maize images:", len(pv_paths), "(expected 3852 colour images)")
    pv_h = hash_many(pv_paths); np.save(PV_CACHE, pv_h)
    print("hashed and cached:", len(pv_h))

## Part 3 (optional, you already have these results)
Unseen-image test on the Kaggle *Corn or Maize Leaf Disease* dataset: 336 images remained after de-duplication and 324 of them were recognised. Set `RUN_PART3 = True` to repeat it.

In [ ]:
RUN_PART3 = False
if RUN_PART3:
    CORN_SLUG = "smaranjitghose/corn-or-maize-leaf-disease-dataset"; CORN_DIR = f"{LOCAL}/corn_candidates"
    if not os.path.isdir(CORN_DIR):
        !kaggle datasets download -d $CORN_SLUG -p $CORN_DIR --unzip -q
    cand = list_images(CORN_DIR); ch = hash_many(cand)
    dist = min_dist_to(pv_h, ch); dup = dist <= THRESH
    print(f"candidates {len(cand)} | near-duplicates of PlantVillage {dup.sum()} | unseen {(~dup).sum()}")
    CAND = pd.DataFrame({"path": cand, "min_hash_dist": dist, "duplicate": dup})
    CAND["label"] = CAND.path.map(lambda s: "healthy" if "healthy" in s.lower() else "diseased")
    un = CAND[~CAND.duplicate].copy(); un["score"] = predict_paths(list(un.path)); un["pred"] = np.where(un.score > 0.5, "healthy", "diseased")
    k = int((un.pred == "diseased").sum()); lo, hi = wilson(k, len(un))
    print(f"recall of diseased on unseen images: {k}/{len(un)} = {k/len(un):.3f}  [{lo:.3f}, {hi:.3f}]")
    CAND.to_csv(f"{OUT_DIR}/field_candidates.csv", index=False); un.to_csv(f"{OUT_DIR}/field_unseen_predictions.csv", index=False)
else:
    print("Part 3 skipped (already done).")

## Part 4. Healthy leaves from other sources (no photos needed from you)

Part 3 could only measure how many *diseased* unseen leaves are caught. To measure **false alarms**, we need healthy maize leaves that PlantVillage does not contain. Several research groups have published field-collected maize datasets with a healthy class, mostly on **Mendeley Data** (not Kaggle):

| Short name | Where | What it has |
|---|---|---|
| `bangladesh_natore_2025` | Mendeley Data, DOI 10.17632/vy629dngm8.1 | Field photos from Natore, Bangladesh: 1,038 healthy, plus gray leaf spot, common rust, bacterial leaf streak, MCMV (the folder also holds augmented copies; the code skips folders named *augment*) |
| `kandahar_afghanistan_2025` | Mendeley Data, DOI 10.17632/zkgh9tddf6.2 | Field photos from four districts of Kandahar, Afghanistan (2025): blight, spot, healthy |
| `corn_leaf_classification_2026` | Mendeley Data, DOI 10.17632/hmkd6nbngr.1 | About 17,000 images, five classes including healthy; collected from field observations *and public repositories*, so some images will be PlantVillage copies (the code removes them) |

The cell below tries to download them through Mendeley's *Download all* link. **Mendeley often blocks scripts (HTTP 403)**; if that happens, open each link above in your browser, click *Download All*, and put the zip files in your Drive folder `crop_yield_project/field_extra_zips/<short name>/` (create the sub-folder). Then re-run the next cell. Large zips take Drive space; delete them afterwards if you need it. Check each dataset's licence on its page and cite it in the paper.

What the cell does: unzip, skip augmented folders, label images from folder names (`healthy` vs the three diseases our classifier knows vs *other* diseases or damage), **remove near-duplicates of PlantVillage**, remove near-duplicates *within* each set, evaluate, and print the false-alarm rate with a Wilson interval. The PlantVillage hashes come from the cache cell above.

In [ ]:
# ---- Download (best effort) to Drive, unpack to fast local disk --------------------------
SOURCES = {"bangladesh_natore_2025": ("vy629dngm8", 1),
           "kandahar_afghanistan_2025": ("zkgh9tddf6", 2),
           "corn_leaf_classification_2026": ("hmkd6nbngr", 1)}
USE_SOURCES = list(SOURCES)      # remove names here to skip a dataset (start with "kandahar_afghanistan_2025" if unsure)

HEADERS = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36", "Accept": "*/*"}

def _stream(url, path):
    with requests.get(url, headers=HEADERS, stream=True, timeout=600, allow_redirects=True) as r:
        r.raise_for_status()
        if "text/html" in r.headers.get("content-type", ""): raise RuntimeError("got a web page instead of a file")
        with open(path, "wb") as fh:
            for ch in r.iter_content(1 << 20): fh.write(ch)

def mendeley_fetch(name, did, ver):
    dest = f"{ZIP_DIR}/{name}"; os.makedirs(dest, exist_ok=True)
    if any(os.scandir(dest)): print(name, ": something already in", dest); return True
    zpath = f"{dest}/{name}.zip"
    try:   # 1) the address behind the "Download all" button
        _stream(f"https://data.mendeley.com/public-api/zip/{did}/download/{ver}", zpath)
        if not zipfile.is_zipfile(zpath): raise RuntimeError("downloaded file is not a zip")
        print(name, ": downloaded", round(os.path.getsize(zpath) / 1e6), "MB"); return True
    except Exception as e:
        print(name, ": 'download all' failed:", e)
        if os.path.exists(zpath): os.remove(zpath)
    try:   # 2) file list through the API, then each file
        r = requests.get(f"https://data.mendeley.com/public-api/datasets/{did}/files?folder_id=root&version={ver}", headers=HEADERS, timeout=60); r.raise_for_status()
        for f in r.json():
            _stream(f["content_details"]["download_url"], f"{dest}/{f.get('filename', 'file.bin')}")
        return True
    except Exception as e:
        print(name, ": file-list download failed:", e)
    print(f"   -> MANUAL STEP for {name}: open https://data.mendeley.com/datasets/{did}/{ver} in your browser, click 'Download all',"
          f"\n      then put the zip into Google Drive at crop_yield_project/field_extra_zips/{name}/  and re-run this cell.")
    return False

for n in USE_SOURCES: mendeley_fetch(n, *SOURCES[n])

for n in USE_SOURCES:
    out = f"{LOCAL}/field_extra/{n}"
    for z in glob.glob(f"{ZIP_DIR}/{n}/**/*.zip", recursive=True):
        mark = f"{out}/.done_{os.path.basename(z)}"
        if not os.path.exists(mark):
            print("unpacking", os.path.basename(z)); os.makedirs(out, exist_ok=True); zipfile.ZipFile(z).extractall(out); open(mark, "w").close()
    imgs = list_images(out) + list_images(f"{ZIP_DIR}/{n}")      # unpacked zips + any images you dropped in directly
    print(n, ": images found:", len(imgs))

In [ ]:
# ---- OPTIONAL: upload a (small) zip from your computer straight into the right Drive folder ----------
# For files above a few hundred MB it is faster to drag the zip into the Drive folder in your browser instead.
UPLOAD_NOW = False
NAME = "kandahar_afghanistan_2025"          # which dataset the zip belongs to
if UPLOAD_NOW:
    from google.colab import files
    os.makedirs(f"{ZIP_DIR}/{NAME}", exist_ok=True)
    for fn, data in files.upload().items():
        open(f"{ZIP_DIR}/{NAME}/{fn}", "wb").write(data); print("saved", fn)

In [ ]:
# ---- Label from folder names, drop augmented folders, de-duplicate ----------------
TRAINED = ["blight", "rust", "gray", "cercospora"]                         # diseases our classifier was trained on
OTHER   = ["streak", "virus", "mottle", "eyespot", "curvularia", "insect", "spot", "disease", "infect", "bacterial", "wilt", "deficien"]

def label3(rel):
    # rel = folder path of the image relative to its dataset root (class folders only, not the dataset's own name)
    txt = rel.lower()
    if "healthy" in txt or "health" in txt: return "healthy"
    if any(k in txt for k in TRAINED):      return "diseased_trained"
    if any(k in txt for k in OTHER):        return "diseased_other"
    return "unknown"

rows = []
for n in USE_SOURCES:
    for root in [f"{LOCAL}/field_extra/{n}", f"{ZIP_DIR}/{n}"]:
        for p in list_images(root):
            rel = os.path.relpath(os.path.dirname(p), root)
            rows.append(dict(source=n, path=p, aug=("augment" in rel.lower()), group=label3(rel), folder=rel))
X = pd.DataFrame(rows)
if X.empty:
    raise SystemExit("No images found. Download at least one dataset first (see the table above).")
print("skipping augmented-folder images:", int(X.aug.sum()), "| unlabeled (no class word in folder name):", int((X.group == "unknown").sum()))
X = X[(~X.aug) & (X.group != "unknown")].reset_index(drop=True)
display(X.groupby(["source", "group"]).size().unstack(fill_value=0))
print("class folders seen (check the labels make sense):"); display(X.groupby(["source", "folder", "group"]).size().rename("n").reset_index().head(40))

# (1) hash everything and remove near-duplicates of PlantVillage training images
hs = hash_many(list(X.path)); X["hash"] = hs
X["min_pv_dist"] = min_dist_to(pv_h, hs)
X["pv_dup"] = X.min_pv_dist <= THRESH
print("near-duplicates of PlantVillage removed:", int(X.pv_dup.sum()))
X = X[~X.pv_dup].reset_index(drop=True)

# (2) remove near-duplicates inside each source (keeps one image per cluster)
keep = np.ones(len(X), bool)
for n, g in X.groupby("source"):
    idx = g.index.values; kept = np.empty(0, dtype=np.uint64)
    for i in idx:
        h = np.uint64(X.hash[i])
        if len(kept) and popcount64(np.bitwise_xor(kept, h)).min() <= THRESH: keep[i] = False
        else: kept = np.append(kept, h)
print("within-set near-duplicates removed:", int((~keep).sum()))
X = X[keep].reset_index(drop=True)

# (3) cap each (source, group) for speed
MAXN = 1000
X = pd.concat([g.sample(min(len(g), MAXN), random_state=SEED) for _, g in X.groupby(["source", "group"])], ignore_index=True)
display(X.groupby(["source", "group"]).size().unstack(fill_value=0))

In [ ]:
# ---- Evaluate: false-alarm rate on healthy leaves ----------------------------------
X["score"] = predict_paths(list(X.path))
X["pred"] = np.where(X.score > 0.5, "healthy", "diseased")

def summarise(df):
    out = []
    for (src, grp), g in df.groupby(["source", "group"]):
        n = len(g)
        k = int((g.pred == "healthy").sum()) if grp == "healthy" else int((g.pred == "diseased").sum())
        lo, hi = wilson(k, n)
        out.append(dict(source=src, group=grp, n=n, correct=k, rate=k / n, ci_lo=lo, ci_hi=hi,
                        metric="specificity (healthy called healthy)" if grp == "healthy" else "recall (diseased called diseased)"))
    return pd.DataFrame(out)

S = summarise(X)
pooled = []
for grp in X.group.unique():
    g = X[X.group == grp]; k = int((g.pred == "healthy").sum()) if grp == "healthy" else int((g.pred == "diseased").sum()); lo, hi = wilson(k, len(g))
    pooled.append(dict(source="ALL", group=grp, n=len(g), correct=k, rate=k / len(g), ci_lo=lo, ci_hi=hi, metric="pooled"))
S = pd.concat([S, pd.DataFrame(pooled)], ignore_index=True)
S.to_csv(f"{OUT_DIR}/field_extra_results.csv", index=False)
X.drop(columns=["hash"]).to_csv(f"{OUT_DIR}/field_extra_predictions.csv", index=False)
display(S.round(3))
h = X[X.group == "healthy"]
print(f"False-alarm rate on unseen healthy field leaves: {1 - (h.pred == 'healthy').mean():.3f}  (n = {len(h)})")

plt.figure(figsize=(5, 3))
for grp, c in [("healthy", "tab:green"), ("diseased_trained", "tab:red")]:
    g = X[X.group == grp]
    if len(g): plt.hist(g.score, bins=20, alpha=.6, color=c, label=f"{grp} (n={len(g)})")
plt.axvline(0.5, color="k", lw=.8); plt.xlabel("P(healthy)"); plt.legend(); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/field_extra_scores.png", dpi=150); plt.show()

## What to send back
The new files in `crop_yield_project/results` on your Drive: `field_extra_results.csv`, `field_extra_predictions.csv` and `field_extra_scores.png` (Part 4). The earlier files you already sent do not need to be sent again.